<a href="https://colab.research.google.com/github/gaurav9479/IIT-Kanpur/blob/main/Drone_Map_IITK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install osmnx folium shapely networkx geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 7.2 MB/s eta 0:00:00


In [ ]:
!pip install osmnx folium shapely networkx geopandas
import osmnx as ox
import folium
import networkx as nx
from shapely.geometry import Polygon
import geopandas as gpd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 3.4 MB/s eta 0:00:00


In [ ]:
place = "Indian Institute of Technology Kanpur, India"

graph = ox.graph_from_place(place, network_type="drive")

print("Map Loaded Successfully")
print("Nodes:", len(graph.nodes))
print("Edges:", len(graph.edges))

Map Loaded Successfully
Nodes: 43
Edges: 94


In [ ]:
nodes, edges = ox.graph_to_gdfs(graph)

nodes.head()

,y,x,street_count,highway,geometry
osmid,,,,,
683475894,26.511809,80.224058,3,NaN,POINT (80.22406 26.51181)
683475928,26.504606,80.230789,4,NaN,POINT (80.23079 26.50461)
683475981,26.509243,80.229108,3,NaN,POINT (80.22911 26.50924)
683476020,26.508073,80.230752,3,crossing,POINT (80.23075 26.50807)
1163366557,26.508083,80.232789,3,NaN,POINT (80.23279 26.50808)


In [ ]:
map_center = [26.5123, 80.2329]

m = folium.Map(
    location=map_center,
    zoom_start=15
)

m

In [ ]:
for idx, row in nodes.iterrows():

    folium.CircleMarker(
        location=(row["y"], row["x"]),
        radius=2,
        color="black"
    ).add_to(m)

m

In [ ]:
hubs = {

    "OAT": [26.5127, 80.2325],

    "Shopping Complex": [26.5098, 80.2317]

}

In [ ]:
for name, coord in hubs.items():

    folium.Marker(
        location=coord,
        popup=name,
        icon=folium.Icon(color="blue")
    ).add_to(m)

m

In [ ]:
zone1 = Polygon([
    (80.2330,26.5120),
    (80.2340,26.5120),
    (80.2340,26.5130),
    (80.2330,26.5130)
])

zone2 = Polygon([
    (80.2290,26.5100),
    (80.2300,26.5100),
    (80.2300,26.5110),
    (80.2290,26.5110)
])

no_fly_zones = [zone1, zone2]

In [ ]:
for zone in no_fly_zones:

    folium.Polygon(
        locations=[(lat,lon) for lon,lat in zone.exterior.coords],
        color="red",
        fill=True,
        fill_opacity=0.4
    ).add_to(m)

m

In [ ]:
lanes = {

    "L1": {
        "altitude":40,
        "direction":"east-west"
    },

    "L2":{
        "altitude":60,
        "direction":"west-east"
    },

    "L3":{
        "altitude":20,
        "type":"emergency"
    }

}

lanes

{'L1': {'altitude': 40, 'direction': 'east-west'},
 'L2': {'altitude': 60, 'direction': 'west-east'},
 'L3': {'altitude': 20, 'type': 'emergency'}}

In [ ]:
vertical_corridor = {

    "radius":20,
    "height":100

}

vertical_corridor

{'radius': 20, 'height': 100}

In [ ]:
m.save("iitk_drone_map.html")

print("Map saved successfully")

Map saved successfully


In [ ]:
print("Total nodes:", len(graph.nodes))
print("Total hubs:", len(hubs))
print("No fly zones:", len(no_fly_zones))
print("Altitude lanes:", lanes)

Total nodes: 43
Total hubs: 2
No fly zones: 2
Altitude lanes: {'L1': {'altitude': 40, 'direction': 'east-west'}, 'L2': {'altitude': 60, 'direction': 'west-east'}, 'L3': {'altitude': 20, 'type': 'emergency'}}


In [ ]:
from shapely.geometry import Point

def is_in_no_fly_zone(lat, lon):

    point = Point(lon, lat)

    for zone in no_fly_zones:
        if zone.contains(point):
            return True

    return False




In [ ]:
print(is_in_no_fly_zone(26.5125, 80.2332))

True


In [ ]:
delivery_points = {

    "Hall1":[26.5160,80.2320],
    "Hall2":[26.5150,80.2340],
    "Library":[26.5070,80.2330],
    "LectureHall":[26.5085,80.2300]

}

delivery_points

{'Hall1': [26.516, 80.232],
 'Hall2': [26.515, 80.234],
 'Library': [26.507, 80.233],
 'LectureHall': [26.5085, 80.23]}

In [ ]:
for name, coord in delivery_points.items():

    folium.Marker(
        location=coord,
        popup=name,
        icon=folium.Icon(color="green")
    ).add_to(m)

m

In [ ]:
lane_colors = {
    "L1":"purple",
    "L2":"orange",
    "L3":"yellow"
}

for lane,color in lane_colors.items():

    folium.Circle(
        location=map_center,
        radius=800,
        color=color,
        fill=False,
        popup=lane
    ).add_to(m)

m

In [ ]:
for hub, coord in hubs.items():

    folium.Circle(
        location=coord,
        radius=20,
        color="blue",
        fill=True,
        fill_opacity=0.2,
        popup=f"{hub} Vertical Corridor"
    ).add_to(m)

m

In [ ]:
import json

infrastructure = {

    "hubs":hubs,
    "delivery_points":delivery_points,
    "lanes":lanes

}

with open("drone_infrastructure.json","w") as f:
    json.dump(infrastructure,f)

print("Infrastructure file saved")

Infrastructure file saved


In [ ]:
m.save("iitk_drone_airspace.html")

print("Final airspace map saved")

Final airspace map saved


In [ ]:
print("Nodes:",len(graph.nodes))
print("Edges:",len(graph.edges))

print("Hubs:",len(hubs))
print("Delivery points:",len(delivery_points))
print("No fly zones:",len(no_fly_zones))

print("Altitude lanes:",lanes)

Nodes: 43
Edges: 94
Hubs: 2
Delivery points: 4
No fly zones: 2
Altitude lanes: {'L1': {'altitude': 40, 'direction': 'east-west'}, 'L2': {'altitude': 60, 'direction': 'west-east'}, 'L3': {'altitude': 20, 'type': 'emergency'}}
